# Data Creation: Loan Default Risk
This notebook handles the full data creation pipeline for the Loan Default Risk project.
It loads the raw CSV dataset from Kaggle, performs basic validation, and inserts the records
into a MongoDB Atlas collection for use in downstream analysis.

## 1. Install Dependencies

In [ ]:
# Install required libraries
!pip install pymongo pandas

## 2. Imports

In [ ]:
import pandas as pd
import logging
from pymongo import MongoClient
from google.colab import userdata, files

# Configure logging so all steps are recorded
logging.basicConfig(
    filename='data_creation.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logging.info('Data creation notebook started')

## 3. Upload Raw Data
Upload the `Loan_default.csv` file downloaded from Kaggle.

**Source:** [Loan Default Dataset on Kaggle](https://www.kaggle.com/datasets/nikhil1e9/loan-default)

In [ ]:
# Upload the CSV file from your local machine
uploaded = files.upload()
logging.info('CSV file uploaded successfully')

## 4. Load and Inspect the Data

In [ ]:
try:
    # Load CSV into a pandas DataFrame
    df = pd.read_csv('Loan_default.csv')
    logging.info(f'CSV loaded successfully with {len(df)} rows and {len(df.columns)} columns')
    print(f'Rows: {len(df)}')
    print(f'Columns: {len(df.columns)}')
    print(f'\nColumn names: {list(df.columns)}')
except Exception as e:
    logging.error(f'Failed to load CSV: {e}')
    raise

In [ ]:
# Preview the first few rows
df.head()

In [ ]:
# Check data types and null counts
df.info()

In [ ]:
# Check for missing values per column
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values found')
logging.info(f'Missing value check complete: {missing.sum()} total missing values')

In [ ]:
# Check distribution of target variable (Default)
print('Default value counts:')
print(df['Default'].value_counts())
print(f'\nDefault rate: {df["Default"].mean()*100:.2f}%')

## 5. Data Validation
Before inserting into MongoDB, we validate that the data meets basic quality requirements.

In [ ]:
try:
    # Validate required columns exist
    required_columns = [
        'LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
        'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
        'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
        'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner', 'Default'
    ]
    missing_cols = [col for col in required_columns if col not in df.columns]
    assert len(missing_cols) == 0, f'Missing columns: {missing_cols}'

    # Validate no duplicate LoanIDs
    assert df['LoanID'].nunique() == len(df), 'Duplicate LoanIDs found'

    # Validate Default column only contains 0 and 1
    assert set(df['Default'].unique()).issubset({0, 1}), 'Default column contains unexpected values'

    print('All validation checks passed!')
    logging.info('All validation checks passed')

except AssertionError as e:
    logging.error(f'Validation failed: {e}')
    raise

## 6. Connect to MongoDB Atlas

In [ ]:
try:
    # Connect using stored secret — never hardcode credentials
    client = MongoClient(
        userdata.get('MONGO_URII'),
        tls=True,
        tlsAllowInvalidCertificates=True
    )

    # Select database (your UVA computing ID) and collection
    db = client['your_uva_id']  # replace with your actual UVA computing ID
    collection = db['loan_default']

    # Confirm connection by listing databases
    print('Connected to MongoDB Atlas')
    print('Available databases:', client.list_database_names())
    logging.info('MongoDB connection established successfully')

except Exception as e:
    logging.error(f'MongoDB connection failed: {e}')
    raise

## 7. Insert Data into MongoDB
We convert the DataFrame to a list of dictionaries (one per document) and insert all records.

In [ ]:
try:
    # Drop collection if it already exists to avoid duplicate inserts on re-runs
    collection.drop()
    logging.info('Existing collection dropped before fresh insert')

    # Convert DataFrame rows to list of dicts for MongoDB insertion
    records = df.to_dict(orient='records')

    # Insert all records at once using insert_many
    result = collection.insert_many(records)

    print(f'Successfully inserted {len(result.inserted_ids)} documents')
    logging.info(f'Inserted {len(result.inserted_ids)} documents into loan_default collection')

except Exception as e:
    logging.error(f'Insert failed: {e}')
    raise

## 8. Verify Insertion

In [ ]:
# Confirm document count matches original CSV row count
count = collection.count_documents({})
print(f'Documents in MongoDB: {count}')
print(f'Rows in CSV:          {len(df)}')
print(f'Match: {count == len(df)}')
logging.info(f'Verification complete: {count} documents in collection')

In [ ]:
# Preview a sample document from MongoDB
print('Sample document from MongoDB:')
print(collection.find_one({}))

## 9. Summary
The raw dataset from Kaggle has been successfully loaded, validated, and inserted into MongoDB Atlas.
The `loan_default` collection now contains **255,347 documents**, each representing one loan applicant
with 18 features including financial background, employment status, and repayment outcome.